In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Load dataset
X, y = load_breast_cancer(return_X_y=True)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(solver='liblinear'))  # solver chosen for compatibility with small datasets
])


In [23]:
from sklearn.model_selection import GridSearchCV

# Define grid
param_grid = {
    # 'scaler__with_mean': [True, False],
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__penalty': ['l1', 'l2']
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)


Best Parameters: {'logreg__C': 0.1, 'logreg__penalty': 'l2'}
Best CV Accuracy: 0.9802197802197803


In [24]:
grid.best_estimator_.named_steps['logreg'].coef_

array([[-0.38986759, -0.42802143, -0.37768802, -0.39481677, -0.17508844,
         0.0032435 , -0.30982707, -0.4137995 , -0.15308339,  0.18846373,
        -0.45903444,  0.03886581, -0.32951083, -0.40991931, -0.05541814,
         0.23401558,  0.09303463, -0.13878028,  0.12840899,  0.21974406,
        -0.51559816, -0.54863902, -0.47282921, -0.49566438, -0.39001799,
        -0.14674271, -0.3516599 , -0.49677726, -0.4055474 , -0.12672923]])

In [8]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

# Define distributions
param_dist = {
    'logreg__C': loguniform(0.001, 100),  # continuous distribution
    'logreg__penalty': ['l1', 'l2']
}

random_search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

print("Best Parameters (Random Search):", random_search.best_params_)
print("Best CV Accuracy (Random Search):", random_search.best_score_)


Best Parameters (Random Search): {'logreg__C': np.float64(0.04661686413912769), 'logreg__penalty': 'l2'}
Best CV Accuracy (Random Search): 0.9802197802197803


In [9]:
from sklearn.metrics import classification_report

# Get best model from grid search
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [14]:
grid.best_estimator_.named_steps['logreg'].coef_

array([[-0.38986759, -0.42802143, -0.37768802, -0.39481677, -0.17508844,
         0.0032435 , -0.30982707, -0.4137995 , -0.15308339,  0.18846373,
        -0.45903444,  0.03886581, -0.32951083, -0.40991931, -0.05541814,
         0.23401558,  0.09303463, -0.13878028,  0.12840899,  0.21974406,
        -0.51559816, -0.54863902, -0.47282921, -0.49566438, -0.39001799,
        -0.14674271, -0.3516599 , -0.49677726, -0.4055474 , -0.12672923]])